# 2段階学習: 分類性能・属性別性能・公平性

次の5条件を同じ test split で比較する。

- ResNet (ERM): ResNet-50 全体を CE で学習した baseline（2段階条件の第1段）
- ResNet + GroupDRO: ResNet-50 全体を最初から age × race の intersectional GroupDRO で学習
- E2E Spatial LoRA (ERM): Spatial LoRA を最初から CE で学習
- E2E Spatial LoRA + GroupDRO: Spatial LoRA を最初から同じ GroupDRO で学習
- 2-stage Spatial LoRA + GroupDRO: ResNet (ERM) の checkpoint から、第2段の Spatial LoRA だけを同じ GroupDRO で学習

- 全体性能: AUROC / BACC / recall_0 / F1 を1図にまとめる
- 属性別性能: age / race / gender / age_group_65 × race ごとに4指標を1図にまとめる
- 公平性: age / race / gender / age_group_65 × race ごとに AUROC gap / BACC gap / Eopp0 / Eopp1 を1図にまとめる

checkpoint は各 run の best validation AUROC を使う。race は既存分析との比較可能性を優先し、White / Asian / Black の3群に限定する。交差属性もこの3 race 群だけを使う。gender はデータ上の sex（Male / Female）を指す。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
from IPython.display import display
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score, roc_auc_score

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import STYLE_SHEET, split_csv
from analysis.common.predictions import cache_path, load_cache

plt.style.use(STYLE_SHEET)
PACKAGE = ROOT / "analysis/two_stage_vs_e2e_age_race_group_dro"
CACHE = PACKAGE / "cache"
RESULTS = PACKAGE / "results"
FIGURES = PACKAGE / "figures"
SPLIT = "test"
MODELS = {
    "ResNet (ERM)": "20260921T103036Z-resnet-chexpert-s42-5538",
    "ResNet + GroupDRO": "20260929T113619Z-resnet-chexpert-age-race-group-dro-s42-5474",
    "E2E Spatial LoRA (ERM)": "20260929T082054Z-spatial-lora-chexpert-s42-b15c",
    "E2E Spatial LoRA + GroupDRO": "20260928T025502Z-spatial-lora-chexpert-age-race-group-dro-s42-9502",
    "2-stage Spatial LoRA + GroupDRO": "20260924T122518Z-spatial-lora-chexpert-from-resnet-age-race-group-dro-s42-985c",
}
METHOD_ORDER = list(MODELS)
COLORS = {
    "ResNet (ERM)": "#4C78A8",
    "ResNet + GroupDRO": "#9ECAE9",
    "E2E Spatial LoRA (ERM)": "#88D27A",
    "E2E Spatial LoRA + GroupDRO": "#54A24B",
    "2-stage Spatial LoRA + GroupDRO": "#F58518",
}
ATTRIBUTES = ["age", "race", "gender", "age_race"]
RACE_LABELS = {0: "White", 2: "Asian", 3: "Black"}
METRICS = ["auroc", "bacc", "recall_0", "f1"]
METRIC_TITLES = {"auroc": "AUROC", "bacc": "BACC", "recall_0": "Recall (class 0)", "f1": "F1"}
FAIRNESS_METRICS = ["auroc_gap", "bacc_gap", "eopp0", "eopp1"]
FAIRNESS_TITLES = {"auroc_gap": "AUROC gap", "bacc_gap": "BACC gap", "eopp0": "Eopp0", "eopp1": "Eopp1"}
GROUP_ORDERS = {
    "age": ["<65", ">=65"],
    "race": ["White", "Asian", "Black"],
    "gender": ["Male", "Female"],
    "age_race": ["<65 × White", "<65 × Asian", "<65 × Black", ">=65 × White", ">=65 × Asian", ">=65 × Black"],
}

## 予測 cache と属性表

予測は `analysis/common/predictions.py` で作成した cache を読む。属性ラベルと予測の行対応は split CSV の行番号で検算する。

In [ ]:
frame = pd.read_csv(split_csv("chexpert", SPLIT))
images = frame["image"].to_numpy(dtype=str)
demographics = pd.DataFrame(
    {
        "age": np.where(frame["age"] >= 65, ">=65", "<65"),
        "race": frame["race"].map(RACE_LABELS),
        "gender": frame["sex"].map({0: "Male", 1: "Female"}),
        "age_race": np.where(frame["age"] >= 65, ">=65", "<65") + " × " + frame["race"].map(RACE_LABELS),
    }
)
missing = pd.DataFrame(
    {
        "age": frame["age_missing"].astype(bool) | frame["age"].isna(),
        "race": frame["race_missing"].astype(bool),
        "gender": frame["sex_missing"].astype(bool),
        "age_race": frame["age_missing"].astype(bool) | frame["age"].isna() | frame["race_missing"].astype(bool),
    }
)
demographics = demographics.mask(missing)
caches = {method: load_cache(cache_path(CACHE, run_id, SPLIT), images) for method, run_id in MODELS.items()}
demographics.notna().sum()

## 指標の集計

`recall_0` は陰性クラスの recall（specificity）、`Eopp0` は群間の recall_0 の最大差、`Eopp1` は群間の陽性 recall の最大差として定義する。gap は小さいほど公平性の観点で望ましい。

In [ ]:
def metrics_for(target, probability, prediction):
    both_classes = (target == 0).any() and (target == 1).any()
    return {
        "auroc": roc_auc_score(target, probability) if both_classes else np.nan,
        "bacc": balanced_accuracy_score(target, prediction) if both_classes else np.nan,
        "recall_0": recall_score(target, prediction, pos_label=0, zero_division=0),
        "f1": f1_score(target, prediction, zero_division=0),
    }


def group_metrics(target, probability, prediction):
    positive, negative = target == 1, target == 0
    result = metrics_for(target, probability, prediction)
    result.update(
        {
            "n": len(target),
            "n_positive": int(positive.sum()),
            "tpr": prediction[positive].mean() if positive.any() else np.nan,
            "tnr": (prediction[negative] == 0).mean() if negative.any() else np.nan,
        }
    )
    return result


performance_rows, subgroup_rows, fairness_rows = [], [], []
for method, cached in caches.items():
    overall = metrics_for(cached["target"], cached["probabilities"][:, 1], cached["predictions"])
    performance_rows.extend(
        {"model": method, "metric": metric, "value": value, "n": len(cached["target"])}
        for metric, value in overall.items()
    )
    for attribute in ATTRIBUTES:
        groups = []
        for group in demographics[attribute].dropna().unique():
            index = demographics.index[demographics[attribute] == group].to_numpy()
            values = group_metrics(
                cached["target"][index], cached["probabilities"][index, 1], cached["predictions"][index]
            )
            groups.append(values)
            subgroup_rows.extend(
                {
                    "model": method,
                    "attribute": attribute,
                    "group": group,
                    "metric": metric,
                    "value": values[metric],
                    "n": values["n"],
                }
                for metric in METRICS
            )
        group_frame = pd.DataFrame(groups)
        fairness = {
            "auroc_gap": group_frame["auroc"].max() - group_frame["auroc"].min(),
            "bacc_gap": group_frame["bacc"].max() - group_frame["bacc"].min(),
            "eopp0": group_frame["tnr"].max() - group_frame["tnr"].min(),
            "eopp1": group_frame["tpr"].max() - group_frame["tpr"].min(),
        }
        fairness_rows.extend(
            {"model": method, "attribute": attribute, "metric": metric, "value": value}
            for metric, value in fairness.items()
        )

classification = pd.DataFrame(performance_rows)
subgroup = pd.DataFrame(subgroup_rows)
fairness = pd.DataFrame(fairness_rows)
RESULTS.mkdir(exist_ok=True)
classification.to_csv(RESULTS / f"classification_performance_{SPLIT}.csv", index=False)
subgroup.to_csv(RESULTS / f"subgroup_performance_{SPLIT}.csv", index=False)
fairness.to_csv(RESULTS / f"fairness_metrics_{SPLIT}.csv", index=False)
classification

## 指定した比較図の作成

各属性図では、x軸に属性群を置き、各群で5手法の bar を横並びにする。公平性図では x軸に4つの公平性指標を置き、各指標で5手法を横並びにする。

In [ ]:
FIGURES.mkdir(exist_ok=True)


def bars_by_group(ax, data, metric, title):
    attribute = data["attribute"].iloc[0]
    observed = set(data["group"])
    groups = [group for group in GROUP_ORDERS[attribute] if group in observed]
    width = 0.8 / len(METHOD_ORDER)
    for i, method in enumerate(METHOD_ORDER):
        values = [
            data[(data["model"] == method) & (data["group"] == group) & (data["metric"] == metric)]["value"].iloc[0]
            for group in groups
        ]
        x = np.arange(len(groups)) + (i - (len(METHOD_ORDER) - 1) / 2) * width
        ax.bar(x, values, width, label=method, color=COLORS[method])
    ax.set_xticks(np.arange(len(groups)), groups)
    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.3)


def plot_subgroup(attribute):
    data = subgroup[subgroup["attribute"] == attribute]
    width = 24 if attribute == "age_race" else 14
    fig, axes = plt.subplots(2, 2, figsize=(width, 8), constrained_layout=True, sharey=True)
    for ax, metric in zip(axes.flat, METRICS):
        bars_by_group(ax, data, metric, METRIC_TITLES[metric])
        ax.tick_params(axis="x", rotation=60 if attribute == "age_race" else 25)
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(f"{attribute} subgroup performance ({SPLIT})")
    fig.savefig(FIGURES / f"{attribute}_subgroup_performance_{SPLIT}.png", dpi=180)
    display(fig)
    plt.close(fig)


def plot_fairness(attribute):
    data = fairness[fairness["attribute"] == attribute]
    fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
    width = 0.8 / len(METHOD_ORDER)
    x = np.arange(len(FAIRNESS_METRICS))
    for i, method in enumerate(METHOD_ORDER):
        values = [
            data[(data["model"] == method) & (data["metric"] == metric)]["value"].iloc[0] for metric in FAIRNESS_METRICS
        ]
        ax.bar(x + (i - (len(METHOD_ORDER) - 1) / 2) * width, values, width, label=method, color=COLORS[method])
    ax.set_xticks(x, [FAIRNESS_TITLES[metric] for metric in FAIRNESS_METRICS])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Gap")
    ax.set_title(f"{attribute} fairness ({SPLIT})")
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8)
    fig.savefig(FIGURES / f"{attribute}_fairness_{SPLIT}.png", dpi=180)
    display(fig)
    plt.close(fig)


fig, axes = plt.subplots(1, 4, figsize=(16, 5), constrained_layout=True, sharey=True)
for ax, metric in zip(axes, METRICS):
    values = [
        classification[(classification["model"] == method) & (classification["metric"] == metric)]["value"].iloc[0]
        for method in METHOD_ORDER
    ]
    ax.bar(METHOD_ORDER, values, color=[COLORS[method] for method in METHOD_ORDER])
    ax.set_title(METRIC_TITLES[metric])
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Score")
fig.suptitle(f"Overall classification performance ({SPLIT})")
fig.savefig(FIGURES / f"overall_classification_performance_{SPLIT}.png", dpi=180)
display(fig)
plt.close(fig)
for attribute in ATTRIBUTES:
    plot_subgroup(attribute)
    plot_fairness(attribute)
sorted(str(path.relative_to(PACKAGE)) for path in FIGURES.glob("*.png"))